# Sprint 1 — Exploração das amostras

Este notebook abre a amostra real do SCR.data que já está em disco e responde
as perguntas da **definição de pronto da Sprint 1** (seção 8 do
`docs/architecture.md`):

1. Qual o encoding do CSV?
2. O separador de campo e o separador decimal são quais?
3. Quantas linhas tem um ano?
4. As 5 colunas que o projeto usa estão presentes?
5. Quantas modalidades começam com `Financiamentos`?
6. `carteira_ativa` está em reais ou em milhares?

Ele **não** faz ingestão. Nada é escrito em disco.

Todos os caminhos e constantes vêm de `src/config.py` — nada chumbado aqui.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd

# Permite importar `src` com o notebook rodando de dentro de notebooks/.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import config

PASTA_AMOSTRAS = config.DIR_BRONZE / "_amostras"
CAMINHO_ZIP = PASTA_AMOSTRAS / "scrdata_2024.zip"

print("Amostra:", CAMINHO_ZIP)
print("Existe? ", CAMINHO_ZIP.exists())
print(f"Tamanho: {CAMINHO_ZIP.stat().st_size / 1024 / 1024:.1f} MiB")

Amostra: /home/yasmin/Documentos/Faculdade/6° Semestre/selic-scr-analysis/data/raw/_amostras/scrdata_2024.zip
Existe?  True
Tamanho: 167.9 MiB


## 1. Encoding, separador de campo e separador decimal

Olhamos os **bytes crus** antes de qualquer `read_csv`. Isso importa: se o
decimal for vírgula e lermos sem `decimal=","`, o pandas devolve o número
errado em silêncio — e a conclusão sobre a unidade sairia errada.

In [2]:
zip_scr = zipfile.ZipFile(CAMINHO_ZIP)
membros = sorted(zip_scr.namelist())

print(f"Membros do ZIP: {len(membros)} (um CSV por mês)")
print("Primeiro:", membros[0], "| Último:", membros[-1])

# Lê um pedaço cru do último mês.
with zip_scr.open(membros[-1]) as f:
    bytes_crus = f.read(2500)

print("\nPrimeiros 4 bytes:", bytes_crus[:4])
print("EF BB BF é o BOM do UTF-8 — é o que exige o 'utf-8-sig'.")

Membros do ZIP: 12 (um CSV por mês)
Primeiro: scrdata_202401.csv | Último: scrdata_202412.csv

Primeiros 4 bytes: b'\xef\xbb\xbfd'
EF BB BF é o BOM do UTF-8 — é o que exige o 'utf-8-sig'.


In [3]:
# Testa os candidatos de config.py NA ORDEM e fica com o primeiro que decodificar.
#
# A ordem importa: "latin-1" aceita qualquer byte e NUNCA falha. Se viesse
# primeiro, venceria sempre e devolveria acento corrompido. Por isso ele é
# o último da lista.
encoding_detectado = None
for candidato in config.SCR_ENCODINGS_CANDIDATOS:
    try:
        texto = bytes_crus.decode(candidato)
        encoding_detectado = candidato
        break
    except UnicodeDecodeError:
        print(f"  {candidato}: falhou")

print("Encoding adotado:", encoding_detectado)

linhas = texto.splitlines()
colunas = linhas[0].split(config.SCR_SEPARADOR)
print(f"\nTotal de colunas: {len(colunas)}")
print("Primeira coluna:", repr(colunas[0]), "<- sem BOM residual")

Encoding adotado: utf-8-sig

Total de colunas: 24
Primeira coluna: 'data_base' <- sem BOM residual


In [4]:
print("As 5 colunas de SCR_COLUNAS_USADAS estão presentes?\n")
for coluna in config.SCR_COLUNAS_USADAS:
    print(f"  {coluna:22} {'SIM' if coluna in colunas else 'NAO ENCONTRADA'}")

print("\nPrimeira linha de dados (crua):\n")
print(linhas[1][:300], "...")

As 5 colunas de SCR_COLUNAS_USADAS estão presentes?

  data_base              SIM
  uf                     SIM
  modalidade             SIM
  numero_de_operacoes    SIM
  carteira_ativa         SIM

Primeira linha de dados (crua):

"2024-12-31";"AC";"Arrendamento";"PJ";"Comércio; reparação de veículos automotores e motocicletas";"Médio";"Operações de arrendamento";"Arrendamento financeiro exceto veículos automotores e imóveis";"Sem destinação específica";"Prefixado";"-1";"159984,06";"420783,77";"851399,05";"0,00";"0,00";"0,00" ...


**Conclusão desta seção**

| Item | Valor confirmado |
|---|---|
| Encoding | `utf-8-sig` (UTF-8 com BOM) |
| Separador de campo | `;` |
| Separador decimal | `,` (vírgula) — visível em `"1432166,88"` |
| Aspas | campos vêm entre aspas duplas |
| Colunas | 24 no total; as 5 que o projeto usa estão todas lá |

## 2. Contagem de linhas do ano

Lemos por **chunks** e só as 5 colunas que interessam. O ano inteiro
descompactado passa de 1 GB — carregar tudo de uma vez estouraria a
memória de uma máquina comum.

In [5]:
LEITURA = dict(
    sep=config.SCR_SEPARADOR,
    encoding=encoding_detectado,
    decimal=",",                      # confirmado na seção 1
    usecols=config.SCR_COLUNAS_USADAS,
)

linhas_por_mes = {}
modalidades = set()

for membro in membros:
    total = 0
    with zip_scr.open(membro) as f:
        for pedaco in pd.read_csv(f, chunksize=100_000, **LEITURA):
            total += len(pedaco)
            modalidades.update(pedaco["modalidade"].unique())
    linhas_por_mes[membro] = total
    print(f"  {membro}: {total:,} linhas")

print(f"\nTOTAL DO ANO: {sum(linhas_por_mes.values()):,} linhas")

  scrdata_202401.csv: 311,692 linhas


  scrdata_202402.csv: 312,030 linhas


  scrdata_202403.csv: 312,636 linhas


  scrdata_202404.csv: 312,358 linhas


  scrdata_202405.csv: 313,841 linhas


...

  scrdata_202408.csv: 310,266 linhas


  scrdata_202409.csv: 309,037 linhas


  scrdata_202410.csv: 308,777 linhas


  scrdata_202411.csv: 308,866 linhas


  scrdata_202412.csv: 310,432 linhas

TOTAL DO ANO: 3,726,515 linhas


## 3. Modalidades

O recorte do projeto é `modalidade LIKE 'Financiamentos%'`. A seção 2.1 do
`architecture.md` diz que são **8** modalidades. Vamos conferir na amostra.

In [6]:
financiamentos = sorted(m for m in modalidades
                        if m.startswith(config.PREFIXO_MODALIDADE))

print(f"Modalidades distintas no ano: {len(modalidades)}")
print(f"Começam com '{config.PREFIXO_MODALIDADE}': {len(financiamentos)}\n")
for m in financiamentos:
    print("  -", m)

print(f"\nEsperado pelo architecture.md: 8 | Encontrado: {len(financiamentos)}")
print("BATE" if len(financiamentos) == 8 else "NAO BATE — anotar a diferença")

Modalidades distintas no ano: 13
Começam com 'Financiamentos': 8

  - Financiamentos
  - Financiamentos com interveniência
  - Financiamentos de infraestrutura e desenvolvimento
  - Financiamentos de títulos e valores mobiliários
  - Financiamentos imobiliários
  - Financiamentos rurais  (ex-financiamentos rurais e agroindustriais)
  - Financiamentos à exportação
  - Financiamentos à importação

Esperado pelo architecture.md: 8 | Encontrado: 8
BATE


In [7]:
print("Todas as modalidades da amostra (para referência):\n")
for m in sorted(modalidades):
    marca = "*" if m.startswith(config.PREFIXO_MODALIDADE) else " "
    print(f" {marca} {m}")
print("\n(* = entra no recorte do projeto)")

Todas as modalidades da amostra (para referência):

   Adiantamentos a depositantes
   Direitos creditórios descontados
   Empréstimos
 * Financiamentos
 * Financiamentos com interveniência
 * Financiamentos de infraestrutura e desenvolvimento
 * Financiamentos de títulos e valores mobiliários
 * Financiamentos imobiliários
 * Financiamentos rurais  (ex-financiamentos rurais e agroindustriais)
 * Financiamentos à exportação
 * Financiamentos à importação
   Operações de arrendamento
   Outros créditos

(* = entra no recorte do projeto)


## 4. `carteira_ativa` está em reais ou em milhares?

A ordem de grandeza decide. Somamos a carteira de **SP** — o maior estado —
em uma modalidade de financiamento, num mês só. Se estiver em reais, o número
fica na casa dos bilhões; se estiver em milhares, ficaria mil vezes menor.

In [8]:
with zip_scr.open(membros[-1]) as f:
    dez = pd.concat(pd.read_csv(f, chunksize=100_000, **LEITURA))

print("Mês analisado:", membros[-1])
print("data_base:", dez["data_base"].unique())

recorte = dez[(dez["uf"] == "SP")
              & (dez["modalidade"].str.startswith(config.PREFIXO_MODALIDADE))]

por_modalidade = (recorte.groupby("modalidade")["carteira_ativa"]
                  .sum()
                  .sort_values(ascending=False))

print("\nCarteira ativa de SP por modalidade de financiamento:\n")
for modalidade, valor in por_modalidade.items():
    print(f"  R$ {valor:>20,.2f}  {modalidade}")

print(f"\nTOTAL SP financiamentos: R$ {por_modalidade.sum():,.2f}")
print(f"Em bilhões: R$ {por_modalidade.sum() / 1e9:,.1f} bi")

Mês analisado: scrdata_202412.csv
data_base: ['2024-12-31']

Carteira ativa de SP por modalidade de financiamento:

  R$   444,848,153,249.60  Financiamentos imobiliários
  R$   293,302,902,042.01  Financiamentos
  R$   109,160,971,459.02  Financiamentos à exportação
  R$    91,321,727,652.54  Financiamentos rurais  (ex-financiamentos rurais e agroindustriais)
  R$    27,120,053,533.33  Financiamentos de infraestrutura e desenvolvimento
  R$     6,483,842,319.45  Financiamentos à importação
  R$     3,077,185,801.44  Financiamentos com interveniência
  R$     1,715,191,059.95  Financiamentos de títulos e valores mobiliários

TOTAL SP financiamentos: R$ 977,030,027,117.34
Em bilhões: R$ 977.0 bi


In [9]:
# `numero_de_operacoes` merece atenção: o BCB usa -1 como MÁSCARA para
# valores abaixo do limite de divulgação. Não é contagem negativa.
print("Valores negativos em numero_de_operacoes:",
      (dez["numero_de_operacoes"] < 0).sum(), "linhas")
print("Valores distintos abaixo de zero:",
      sorted(dez.loc[dez["numero_de_operacoes"] < 0,
                     "numero_de_operacoes"].unique()))
print("\n-1 = máscara do BCB, não contagem. Tratar na Silver (Sprint 4).")

Valores negativos em numero_de_operacoes: 83511 linhas
Valores distintos abaixo de zero: [np.int64(-1)]

-1 = máscara do BCB, não contagem. Tratar na Silver (Sprint 4).


**Conclusão desta seção**

O total de SP em financiamentos fecha em **R$ 977,0 bilhões** em dez/2024. Isso
só faz sentido se os valores estiverem em **reais**: em milhares, implicaria uma
carteira mil vezes maior, incompatível com o crédito brasileiro.

> **`carteira_ativa` está em REAIS.**

E `numero_de_operacoes` usa **`-1` como máscara** para valores abaixo do limite
de divulgação do BCB — não é contagem negativa. São **83.511 linhas de 310.432**
(27%) só em dez/2024, então não é caso raro: precisa ser tratado na Silver.

---

## Resumo — respostas da definição de pronto da Sprint 1

Amostra: `scrdata_2024.zip`, coletada em **2026-09-03**.

| Pergunta | Resposta |
|---|---|
| Encoding | `utf-8-sig` (UTF-8 com BOM) |
| Separador de campo | `;` |
| Separador decimal | `,` (vírgula) |
| Tamanho do ZIP | 167,9 MiB — 12 CSVs mensais, ~1,15 GB descompactado |
| Linhas em 2024 | **3.726.515** (~310 mil por mês) |
| Colunas | 24; as 5 de `SCR_COLUNAS_USADAS` todas presentes |
| Modalidades no total | 13 |
| Modalidades `Financiamentos%` | **8** — bate com a seção 2.1 do `architecture.md` |
| Unidade de `carteira_ativa` | **reais** |
| Armadilha encontrada | `numero_de_operacoes = -1` é máscara, em 27% das linhas |

### Dois detalhes para a Sprint 2

1. O nome exato de uma modalidade é
   `Financiamentos rurais  (ex-financiamentos rurais e agroindustriais)` —
   com **dois espaços** antes do parêntese. O filtro por prefixo não se
   incomoda, mas qualquer comparação por igualdade precisa do nome exato.
2. A amostra é de **2024**. Os anos anteriores podem ter layout diferente —
   a Sprint 2 precisa conferir o cabeçalho de cada ano, não assumir este.